# Õppetund 08 - Mitme Agendi Disainimuster


## Paigaldus


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Miks kasutatakse mitmeagentseid süsteeme?

Reaalmaailma ülesanded, nagu reisiplaanide koostamine, nõuavad mitmesuguseid teadmisi — logistikat, kohalikku teadmist, eelarvestamist ja palju muud. Üks agent, kes püüab kõike ise hallata, muutub kiiresti keeruliseks ja raskesti juhitavaks.

Mitmeagentsetes süsteemides seda probleemi lahendatakse **spetsialiseerumise** kaudu: iga agent keskendub ühele valdkonnale, saavutades paremaid tulemusi kui üldistaja. Need parandavad ka **skaalautuvust** — uute agentide lisamine (nt lennuspetsialist, restoranikriitik) on võimalik ilma olemasolevat tööd ümber kirjutamata. Agendid töötavad koos struktureeritud torujuhtme kaudu, edastades konteksti ühest teisele.


## Spetsialiseeritud agentide loomine


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Jada töövoo loomine

`WorkflowBuilder` võimaldab ühendada agendid suunatud graafikusse. Siin loome lihtsa kaheastmelise torujuhtme: **TravelPlanner** koostab marsruudi kavandi ja seejärel **TravelConcierge** vaatab selle üle ja täiustab seda.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Rohkem agente töösse lisamine

Üks multi-agentse mustri suurimaid eeliseid on selle lihtne laiendatavus. Allpool lisame **BudgetReviewer** agendi, kes kontrollib plaani reisihaalduse eelarve suhtes, märgib esemed, mis võivad kulusid üle piiri ajada, ning pakub raha säästvaid alternatiive. Töös korraldatakse nüüd kolme agendi järjestikune käivitamine:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Kokkuvõte

Selles õppetükis õppisite, kuidas:

1. **Luua spetsialiseeritud agente** — igaühel on keskendunud roll (planeerimine, teenindaja, eelarve ülevaatus).
2. **Ühenduda agendid järjestikuseks töövooguks** kasutades `WorkflowBuilder` ja `add_edge`.
3. **Voogesitada väljundit** mitme agendi toru kaudu, jälgides, milline agent parasjagu räägib.
4. **Laiendada töövoogu**, lisades ahelasse uusi agente ilma olemasolevaid muutmata.

Mitme agendi disainimuster hoiab iga agendi lihtsana, samal ajal tootes rikkalikumaid ja põhjalikumalt läbi vaadatud tulemusi kui ükski üksik agent üksi suudaks.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tähelepanu**:  
See dokument on tõlgitud tehisintellekti tõlketeenuse [Co-op Translator](https://github.com/Azure/co-op-translator) abil. Kuigi püüame tagada täpsust, palun arvestage, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument oma emakeeles tuleks pidada usaldusväärseks allikaks. Olulise teabe puhul soovitame kasutada professionaalset inimtõlget. Me ei vastuta ühegi arusaamatuse või valesti mõistmise eest, mis võib tuleneda selle tõlke kasutamisest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
